Importing libraries

In [1]:
import librosa
import soundfile
import os, glob, pickle
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

Feature Extraction

In [2]:
def extract_feature(file_name, mfcc=True, chroma=True, mel=True, contrast=True, tonnetz=True, zcr=True):
    with soundfile.SoundFile(file_name) as sound_file:
        X = sound_file.read(dtype="float32")
        sample_rate = sound_file.samplerate
        result = np.array([])

        if chroma or contrast:
            stft = np.abs(librosa.stft(X))

        if mfcc:
            mfccs = np.mean(librosa.feature.mfcc(y=X, sr=sample_rate, n_mfcc=40).T, axis=0)
            result = np.hstack((result, mfccs))
        if chroma:
            chroma = np.mean(librosa.feature.chroma_stft(S=stft, sr=sample_rate).T, axis=0)
            result = np.hstack((result, chroma))
        if mel:
            mel = np.mean(librosa.feature.melspectrogram(y=X, sr=sample_rate).T, axis=0)
            result = np.hstack((result, mel))
        if contrast:
            contrast = np.mean(librosa.feature.spectral_contrast(S=stft, sr=sample_rate).T, axis=0)
            result = np.hstack((result, contrast))
        if tonnetz:
            tonnetz = np.mean(librosa.feature.tonnetz(y=librosa.effects.harmonic(X), sr=sample_rate).T, axis=0)
            result = np.hstack((result, tonnetz))
        if zcr:
            zcr = np.mean(librosa.feature.zero_crossing_rate(y=X).T, axis=0)
            result = np.hstack((result, zcr))

    return result


Load dataset and extract features from each sound file

In [3]:
def load_data(test_size=0.25):
    x, y = [], []
    emotions = {'01': 'neutral', '02': 'calm', '03': 'happy', '04': 'sad', 
                '05': 'angry', '06': 'fearful', '07': 'disgust', '08': 'surprised'}
    
    # Path to dataset folder
    dataset_path = 'E:\Programs\Python\Projects\speech-emotion-recognition\Dataset'

    # Loop through actor folders
    for actor_folder in os.listdir(dataset_path):
        actor_path = os.path.join(dataset_path, actor_folder)
        if os.path.isdir(actor_path):
            # Loop through wav files in the actor folder
            for file in glob.glob(os.path.join(actor_path, '*.wav')):
                # Extract features
                features = extract_feature(file, mfcc=True, chroma=True, mel=True)
                
                # Get the emotion label from the file name
                file_name = os.path.basename(file)
                emotion_code = file_name.split("-")[2]  # Assuming emotion code is the third element
                emotion = emotions.get(emotion_code)
                
                if emotion:
                    x.append(features)
                    y.append(emotion)

    # Convert the data into numpy arrays
    x = np.array(x)
    y = np.array(y)

    # Split the dataset into training and testing sets
    return train_test_split(x, y, test_size=test_size, random_state=42)


In [4]:
# Split the dataset
x_train, x_test, y_train, y_test = load_data(test_size=0.25)

# Get the shape of the training and testing datasets
print((x_train.shape[0], x_test.shape[0]))

# Get the number of features extracted
print(f'Features extracted: {x_train.shape[1]}')

# Initialize the Multi-Layer Perceptron Classifier
model = MLPClassifier(alpha=0.01, batch_size=256, epsilon=1e-08, hidden_layer_sizes=(300,), learning_rate='adaptive', max_iter=500)

# Train the model
model.fit(x_train, y_train)

# Predict for the test set
y_pred = model.predict(x_test)

# Calculate the accuracy of our model
accuracy = accuracy_score(y_true=y_test, y_pred=y_pred)

# Print the accuracy
print("Accuracy: {:.2f}%".format(accuracy * 100))

(1080, 360)
Features extracted: 194
Accuracy: 46.39%
